In [1]:
!pip install torch transformers datasets peft accelerate bitsandbytes


In [2]:
from google.colab import files
uploaded = files.upload()

Saving dataset.jsonl to dataset (1).jsonl


In [2]:
!ls /content


dataset.jsonl  deepseek_finetuned  deepseek_finetuned_full  sample_data


In [3]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, pipeline,BitsAndBytesConfig
)
from peft import get_peft_model, LoraConfig, TaskType, PeftModel

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:
dataset_path = "/content/dataset.jsonl"

# Load dataset
dataset = load_dataset("json", data_files=dataset_path, split="train")
train_test_split = dataset.train_test_split(test_size=0.1) #10% data will be used for evaluation(test)
train_dataset, eval_dataset = train_test_split["train"], train_test_split["test"]

# Load pre-trained model and tokenizer
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenization function
def tokenize_function(examples):
    combined_texts = [f"{p}\n{c}" for p, c in zip(examples["prompt"], examples["completion"])]
    tokenized = tokenizer(combined_texts, truncation=True, max_length=512, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Tokenize datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Load model with optimized memory usage
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,  # Use 8-bit mode
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config
)

# Apply LoRA (Low-Rank Adaptation) for efficient fine-tuning
lora_config = LoraConfig(r=4, lora_alpha=8, lora_dropout=0.1, task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Show trainable parameters

# Define training arguments
training_args = TrainingArguments(
    output_dir="deepseek_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_steps=100,
    evaluation_strategy="epoch",
    learning_rate=3e-5,
    logging_dir="./logs",
    report_to="none",
)

# Initialize Trainer and start fine-tuning
trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=eval_dataset)
trainer.train()

# Save fine-tuned model
save_path = "deepseek_finetuned"
os.makedirs(save_path, exist_ok=True)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Merge LoRA adapters into base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True
)
model = PeftModel.from_pretrained(base_model, save_path).merge_and_unload()

# Save final merged model
final_save_path = "deepseek_finetuned_full"
os.makedirs(final_save_path, exist_ok=True)
model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

`low_cpu_mem_usage` was None, now default to True since model is quantized.


trainable params: 544,768 || all params: 1,777,632,768 || trainable%: 0.0306


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
0,No log,10.713919


('deepseek_finetuned_full/tokenizer_config.json',
 'deepseek_finetuned_full/special_tokens_map.json',
 'deepseek_finetuned_full/tokenizer.json')

In [5]:

# Load fine-tuned model for text generation
model = AutoModelForCausalLM.from_pretrained(final_save_path, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32).to(device)
tokenizer = AutoTokenizer.from_pretrained(final_save_path)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

Device set to use cuda:0


In [19]:
from pprint import pprint

# Generate text
generated_text = pipe("What is AI?", max_length=400, num_return_sequences=1)
pprint(generated_text[0]['generated_text'])

('What is AI? What is the difference between AI and machine learning? What is '
 'the difference between AI and deep learning?\n'
 '\n'
 'I need to write a detailed answer for this in an exam. Can I get a copy of '
 "the answer? Or should I think it through myself? I'm a bit confused.\n"
 'Okay, so I need to understand AI, machine learning, and deep learning. Hmm, '
 'where to start. AI, as I remember, is about creating systems that can '
 'perform tasks that typically require human intelligence. Like, if I can '
 "write a program that can recognize images or translate languages, that's AI. "
 "But wait, isn't machine learning a part of AI? So maybe AI is broader than "
 'just machine learning.\n'
 '\n'
 'Machine learning, if I recall correctly, is a subset of AI that focuses on '
 "developing algorithms that can learn from and make use of data. So it's "
 'about the ability of machines to improve through experience. For example, a '
 'machine learning model could be trained on a datas